In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG ecommerce_project")
spark.sql("USE SCHEMA sales")

In [0]:
%run ./00_transformations

In [0]:
raw_orders = spark.table("raw_orders")
enriched_orders = build_enriched_orders(
    raw_orders, spark.table("enriched_customers"), spark.table("enriched_products")
)
save_table(enriched_orders, "enriched_orders", expected_min_rows=9994)

In [0]:
e = spark.table("enriched_orders")
r = spark.table("raw_orders")

print("rows enriched / raw :", e.count(), r.count())                       # both 9994
print("null dates          :", e.filter("order_date IS NULL").count())     # 0
print("years               :", sorted([x[0] for x in e.select("year").distinct().collect()]))  # 2014-2017
print("null category rows  :", e.filter("category IS NULL").count())       # 204
print("profit on null-category rows:", e.filter("category IS NULL").agg(F.round(F.sum("profit"), 2)).first()[0])
print("profit enriched/raw :", e.agg(F.round(F.sum("profit"), 2)).first()[0],
                               r.agg(F.round(F.sum("profit"), 2)).first()[0])
display(e.limit(5))